# Qwen-Image-2.1-Turbo 在 Google Colab 上运行

基于 [abenzerps/Qwen-Image-2.1-Turbo-Quantized](https://huggingface.co/abenzerps/Qwen-Image-2.1-Turbo-Quantized)，
在 Colab 免费 T4 (16GB VRAM) 上部署，并提供 **OpenAI 风格 `/v1/images/generations` 接口**（端口 `8010`），
再用 `cloudflared` 暴露公网地址（固定域名见文末可选单元格）。

## 模型选型（Colab 推荐）

| 量化文件 | 大小 | 建议 |
|---|---|---|
| `qwen-image-2.1-turbo-Q4_K_M.gguf` | 4.05 GB | ✅ Colab 首选，体积与质量平衡 |
| `qwen-image-2.1-turbo-Q4_0.gguf` | 4.05 GB | 显存极紧时的备选 |
| `qwen-image-2.1-turbo-Q5_K_M.gguf` | 4.94 GB | 质量更好，稍慢，T4 16GB 可跑 |
| `qwen-image-2.1-turbo-Q6_K.gguf` | 5.88 GB | 显存吃紧，慎选 |
| `qwen-image-2.1-turbo-Q8_0.gguf` | 7.59 GB | 一般不建议在免费 Colab 上使用 |

必须配套下载（工作流三件套）：
- 文本编码器：`text_encoders/qwen3vl_8b_int8_convrot.safetensors` (9.35 GB，INT8 更省显存)
- VAE：`vae/qwen_image_2.1_vae_bf16.safetensors` (0.68 GB)

合计约 **14 GB** 下载；Colab 免费版磁盘通常充足，下载需 10–30 分钟，视网络而定。

> **注意**：这是文生图（DiT 7B）模型，**不适用 `llama-cpp-python`**（那是文本模型推理库）。
本笔记本采用 ComfyUI + [ComfyUI-GGUF](https://github.com/leejet/ComfyUI-GGUF) 运行 GGUF 权重。

**关于 FastAPI 这层**：ComfyUI 自带完整 HTTP API（`/prompt`、`/history`、`/view`），本身就能出图，
FastAPI **不是必需的**，它只是把 ComfyUI 的工作流 JSON 包装成 OpenAI 兼容的
`/v1/images/generations` + `b64_json`，方便沿用 OpenAI 客户端。
若不想要这层，把隧道 Cell 里的 `TUNNEL_PORT` 改成 `8188`，直接访问 ComfyUI 原生 API 即可。

运行前：`Runtime → Change runtime type → T4 GPU`，保存后重新运行全部 Cell。


In [ ]:
import sys, os
print('Python:', sys.version)
try:
    import torch
    print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
except Exception as e:
    print('torch 未安装:', e)
!nvidia-smi
!df -h /content | tail -1


In [ ]:
import urllib.request, time
# Colab 机器在境外：直连官方 huggingface.co / pypi.org 最全最快。
# 国内镜像常未同步 2026 年的新包/新仓库，不要用。
REPO_PATH = '/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main'
REMOTES = [
    'qwen-image-2.1-turbo-Q4_K_M.gguf',
    'text_encoders/qwen3vl_8b_int8_convrot.safetensors',
    'vae/qwen_image_2.1_vae_bf16.safetensors',
]
CANDIDATES = [
    ("huggingface.co 官方", "https://huggingface.co"),
    ("hf-mirror.com 备用", "https://hf-mirror.com"),
]

def head(base, path):
    req = urllib.request.Request(base + REPO_PATH + "/" + path, method="HEAD")
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            return round(time.time() - t0, 2), int(r.headers.get('Content-Length', 0)), r.status
    except Exception as e:
        return None, 0, repr(e)[:80]

# 用最小的 VAE 探测延迟，避免大文件 HEAD 拖慢
rows = []
for name, base in CANDIDATES:
    dt, size, status = head(base, REMOTES[2])
    rows.append((name, base, dt, size, status))
    print(f'{name:24s} {str(dt):>6}s  status={status}  probe={size/1e6:.1f} MB')

ok = [r for r in rows if r[2] is not None and r[3] > 0]
BASE = min(ok, key=lambda r: r[2])[1] if ok else "https://huggingface.co"
print(">>> 选用源:", BASE)

# 关键：远端路径带子目录，拿真实大小作为下载校验基准
EXPECT = {}
for p in REMOTES:
    _, sz, _ = head(BASE, p)
    EXPECT[p] = sz
    print(f'  远端 {p}: {sz/1e9:.2f} GB' if sz else f'  ⚠️ 取不到大小: {p}')


In [ ]:
import os, subprocess, sys
os.chdir('/content')
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
os.chdir('/content/ComfyUI')
PYPI = 'https://pypi.org/simple'
# 全量安装官方 requirements：含 comfy-aimdo==0.6.0 与 comfy-kitchen==0.2.41
# 这两个是 2026 新包，缺了会让 comfy/quant_ops.py 的 ck 未定义，ComfyUI 直接崩
!pip install -q -r requirements.txt -i $PYPI
!pip install -q fastapi uvicorn requests -i $PYPI
# 精确兜底：INT8 ConvRot 文本编码器依赖 comfy-kitchen 的量化 layout
!pip install -q 'comfy-kitchen==0.2.41' 'comfy-aimdo==0.6.0' -i $PYPI
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/leejet/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
!pip install -q -r custom_nodes/ComfyUI-GGUF/requirements.txt -i $PYPI
# 装完立刻校验，避免静默漏装
chk = subprocess.run([sys.executable, '-c',
    'import comfy_kitchen, comfy_aimdo; from comfy.quant_ops import ck; print("依赖 OK")'],
    capture_output=True, text=True)
print('依赖校验:', (chk.stdout or chk.stderr).strip()[-400:])
print('ComfyUI 就绪')


In [ ]:
import os, time
MODEL_DIR = '/content/ComfyUI/models'
# 左：本地路径；右：HF 仓库内路径（text_encoders/ 与 vae/ 是仓库里的子目录，不能省）
FILES = [
    ('diffusion_models/qwen-image-2.1-turbo-Q4_K_M.gguf', 'qwen-image-2.1-turbo-Q4_K_M.gguf'),
    ('text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'text_encoders/qwen3vl_8b_int8_convrot.safetensors'),
    ('vae/qwen_image_2.1_vae_bf16.safetensors', 'vae/qwen_image_2.1_vae_bf16.safetensors'),
]
for rel, remote in FILES:
    dst = os.path.join(MODEL_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    expect = EXPECT.get(remote, 0)
    if os.path.exists(dst) and (expect == 0 or os.path.getsize(dst) == expect):
        print('✅ 已完整，跳过:', rel, '%.2f GB' % (os.path.getsize(dst)/1e9))
        continue
    if os.path.exists(dst) and os.path.getsize(dst) < 1e6:
        os.remove(dst)   # 清掉 404 留下的 0 字节残留
    url = BASE + '/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/' + remote
    print('下载:', remote)
    t0 = time.time()
    !wget -c -q --show-progress "$url" -O "$dst"
    dt = max(time.time() - t0, 1e-6)
    got = os.path.getsize(dst) if os.path.exists(dst) else 0
    ok = (expect == 0 or abs(got - expect) < expect * 0.01) and got > 1e6
    print(('✅' if ok else '❌'), rel, '%.2f GB' % (got/1e9), '| %.1f MB/s' % (got/1e6/dt))


In [ ]:
import subprocess, sys, time, socket, os

def port_open(p, host='127.0.0.1'):
    s = socket.socket(); s.settimeout(1)
    try:
        s.connect((host, p)); return True
    except Exception:
        return False
    finally:
        s.close()

if port_open(8188):
    print('ComfyUI 已在运行: http://127.0.0.1:8188')
else:
    # 用 w 覆盖，避免旧 traceback 混进日志误导排查
    log = open('/content/comfyui.log', 'w')
    comfy = subprocess.Popen([sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
                             cwd='/content/ComfyUI', stdout=log, stderr=log, start_new_session=True)
    print('ComfyUI PID:', comfy.pid)
    for _ in range(60):
        time.sleep(3)
        if port_open(8188):
            break
    if port_open(8188):
        print('✅ ComfyUI 已就绪: http://127.0.0.1:8188')
        try:
            import urllib.request, json
            info = json.load(urllib.request.urlopen('http://127.0.0.1:8188/object_info', timeout=30))
            print('  UnetLoaderGGUF 已注册:', 'UnetLoaderGGUF' in info)
        except Exception as e:
            print('  object_info 读取失败:', repr(e))
    else:
        print('❌ 启动失败，日志末尾 40 行:' )
        print('\n'.join(open('/content/comfyui.log').read().splitlines()[-40:]))


In [ ]:
%%writefile /content/qwen_api.py
# OpenAI 兼容代理：把 ComfyUI 的 Qwen-Image 2.1 工作流包装成 /v1/images/generations
# 节点结构按 ComfyUI 官方 image_qwen_image_2_1_t2i 工作流 + ComfyUI-GGUF 修正：
#   UnetLoaderGGUF / CLIPLoader(type=qwen_image) / VAELoader
#   TextEncodeQwenImage21（同时输出 positive / negative / 64 通道空 latent）
#   KSampler / VAEDecode / SaveImage
from fastapi import FastAPI
from pydantic import BaseModel
import requests, base64, uuid, time, uvicorn, traceback

app = FastAPI(title='Qwen-Image-2.1-Turbo proxy')
COMFY = 'http://127.0.0.1:8188'
UNET = 'qwen-image-2.1-turbo-Q4_K_M.gguf'
CLIP = 'qwen3vl_8b_int8_convrot.safetensors'
VAE = 'qwen_image_2.1_vae_bf16.safetensors'
UNET_CLASS = 'UnetLoaderGGUF'

class ImageGenRequest(BaseModel):
    prompt: str
    model: str = UNET
    width: int = 1024
    height: int = 1024
    steps: int = 8
    seed: int = -1
    cfg: float = 1.0
    negative_prompt: str = ''

@app.get('/')
def root():
    return {'status': 'ok', 'backend': COMFY, 'endpoint': '/v1/images/generations'}

@app.get('/health')
def health():
    try:
        requests.get(COMFY + '/system_stats', timeout=5).raise_for_status()
        return {'ok': True, 'comfyui': 'up'}
    except Exception as e:
        return {'ok': False, 'comfyui': repr(e)}

@app.get('/v1/models')
def models():
    return {'object': 'list', 'data': [{'id': UNET, 'object': 'model'}]}

def build_workflow(req: ImageGenRequest):
    seed = req.seed if req.seed and req.seed >= 0 else uuid.uuid4().int >> 64
    # Qwen-Image 2.1 的 latent 是 64 通道、边长 /16，由 TextEncodeQwenImage21 生成，
    # 不能用 EmptyLatentImage（那是 4 通道、边长 /8 的老格式）。
    # 该节点无参考图时输出正方形 latent，边长由 resolution 决定。
    resolution = max(32, (max(req.width, req.height) // 32) * 32)
    return {
        '1': {'inputs': {'unet_name': req.model}, 'class_type': UNET_CLASS},
        '2': {'inputs': {'clip_name': CLIP, 'type': 'qwen_image'}, 'class_type': 'CLIPLoader'},
        '3': {'inputs': {'vae_name': VAE}, 'class_type': 'VAELoader'},
        '4': {'inputs': {
            'clip': ['2', 0],
            'prompt': req.prompt,
            'negative_prompt': req.negative_prompt,
            'vae': ['3', 0],
            'resolution': resolution,
        }, 'class_type': 'TextEncodeQwenImage21'},
        '5': {'inputs': {
            'model': ['1', 0],
            'positive': ['4', 0],
            'negative': ['4', 1],
            'latent_image': ['4', 2],
            'seed': int(seed),
            'steps': req.steps,
            'cfg': req.cfg,
            'sampler_name': 'euler',
            'scheduler': 'normal',
            'denoise': 1.0,
        }, 'class_type': 'KSampler'},
        '6': {'inputs': {'samples': ['5', 0], 'vae': ['3', 0]}, 'class_type': 'VAEDecode'},
        '7': {'inputs': {'images': ['6', 0], 'filename_prefix': 'qwen_colab'}, 'class_type': 'SaveImage'},
    }

@app.post('/v1/images/generations')
def generate(req: ImageGenRequest):
    try:
        r = requests.post(COMFY + '/prompt', json={'prompt': build_workflow(req)}, timeout=30)
        if r.status_code != 200:
            return {'error': 'comfyui /prompt rejected', 'detail': r.text[:1500]}
        pid = r.json()['prompt_id']
        outs = None
        for _ in range(240):
            time.sleep(2)
            h = requests.get(COMFY + f'/history/{pid}', timeout=10).json()
            if pid in h and 'outputs' in h[pid]:
                outs = h[pid]['outputs']
                break
            if pid in h and h[pid].get('status', {}).get('status_str') == 'error':
                return {'error': 'comfyui execution error', 'detail': str(h[pid].get('status'))[:1500]}
        if outs is None:
            return {'error': 'generation timeout'}
        images = []
        for node_id, out in outs.items():
            for img in out.get('images', []):
                data = requests.get(COMFY + '/view',
                    params={'filename': img['filename'], 'subfolder': img.get('subfolder', ''), 'type': 'output'},
                    timeout=60).content
                images.append({'b64_json': base64.b64encode(data).decode('ascii')})
        if not images:
            return {'error': 'no images in output', 'detail': str(outs)[:1500]}
        return {'created': int(time.time()), 'data': images}
    except Exception:
        return {'error': 'proxy exception', 'detail': traceback.format_exc()[-1500:]}

if __name__ == '__main__':
    print('qwen_api: listening on 0.0.0.0:8010 -> ' + COMFY, flush=True)
    uvicorn.run(app, host='0.0.0.0', port=8010, log_level='info')


In [ ]:
import subprocess, sys, time, os

if not os.path.exists('/content/qwen_api.py'):
    raise RuntimeError('缺少 /content/qwen_api.py —— 请先运行上一单元格（%%writefile）')

# 清掉可能残留的死进程，避免端口占用
subprocess.run("ps -eo pid,args | grep '[q]wen_api.py' | awk '{print $1}' | xargs -r kill", shell=True)
time.sleep(1)
api = subprocess.Popen([sys.executable, 'qwen_api.py'], cwd='/content',
                       stdout=open('/content/qwen_api.log', 'w'), stderr=subprocess.STDOUT,
                       start_new_session=True)
print('FastAPI 代理 PID:', api.pid)
for _ in range(40):
    if port_open(8010):
        break
    time.sleep(2)
if port_open(8010):
    print('✅ FastAPI 已就绪: http://127.0.0.1:8010/v1/images/generations')
else:
    print('❌ 8010 未监听，/content/qwen_api.log 末尾:' )
    print(open('/content/qwen_api.log').read()[-2000:])


In [ ]:
import subprocess, re, time, os, urllib.request

TUNNEL_PORT = 8010   # 改成 8188 即直连 ComfyUI 原生 API（不需要 FastAPI 这层）

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

running = subprocess.run("ps -eo args | grep -c '[c]loudflared tunnel'", shell=True,
                         capture_output=True, text=True).stdout.strip()
if running != '0':
    print('已有 cloudflared 隧道进程，保持原公网域名不变；后端恢复后自动解除 502')
else:
    tlog = open('/content/cloudflared.log', 'w')
    subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{TUNNEL_PORT}'],
                     stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
    public = None
    for _ in range(40):
        time.sleep(2)
        txt = open('/content/cloudflared.log').read()
        m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
        if m:
            public = m[0]
            break
    print('>>> 公网地址:', public)
    if TUNNEL_PORT == 8010 and public:
        print('>>> OpenAI 兼容接口:', public + '/v1/images/generations')
    elif public:
        print('>>> ComfyUI 原生 API:', public + '/system_stats')


In [ ]:
import requests, base64, time
from IPython.display import Image, display
payload = {
    'prompt': '一只穿宇航服的猫坐在月球表面，背景是地球升起，电影级光照，超细节，4K',
    'width': 1024, 'height': 1024, 'steps': 8, 'seed': 42, 'cfg': 1.0
}
t0 = time.time()
r = requests.post('http://127.0.0.1:8010/v1/images/generations', json=payload, timeout=600)
dt = time.time() - t0
print('HTTP', r.status_code, '| 端到端耗时: %.1f 秒' % dt)
res = r.json()
if 'error' in res:
    print('❌', res.get('error'), '\n', res.get('detail', '')[:1500])
else:
    img_b64 = res['data'][0]['b64_json']
    open('/content/qwen_image_test.png', 'wb').write(base64.b64decode(img_b64))
    display(Image(filename='/content/qwen_image_test.png'))


# 可选：固定私有域名 + 8188 ComfyUI 访问

主流程的 cloudflared 是 **quick tunnel**，域名每次重启 Colab 都会变。下面两个单元格启用固定域名，
并把 8188（ComfyUI Web UI / 原生 API）也单独暴露出来。

| 域名 | 后端 | 用途 |
| --- | --- | --- |
| `qwythos.gmhqyj.ccwu.cc` | `localhost:8010` | OpenAI 兼容 `/v1/images/generations` |
| `comfy.gmhqyj.ccwu.cc` | `localhost:8188` | ComfyUI Web UI、`/prompt`、`/history`、`/view`、`/object_info` |

**前提**：Colab 左侧 Secrets 已添加 `CF_TUNNEL_TOKEN` 并开启本笔记本访问权限；Cloudflare Zero Trust
→ Networks → Tunnels → 该隧道 → Public Hostname 已按上表配好两条。

⚠️ 两个域名目前都没有鉴权，8188 的 `/view` 能读输出文件，建议加 Cloudflare Access。
若本机浏览器打不开 `*.gmhqyj.ccwu.cc`（部分网络会在 TLS 握手阶段阻断），用第三个单元格给出的
`trycloudflare.com` 临时域名，或换网络 / 调整代理规则。


In [ ]:
# 固定私有域名：启动 Cloudflare Named Tunnel（token 从 Colab Secret 读取，笔记本内无明文）
import os, subprocess, time, urllib.request
from google.colab import userdata

try:
    TOKEN = userdata.get('CF_TUNNEL_TOKEN')
except Exception as e:
    raise RuntimeError('请先在 Colab 左侧 Secrets 添加 CF_TUNNEL_TOKEN，并开启本笔记本访问权限') from e

CF = '/usr/local/bin/cloudflared'
if not os.path.exists(CF):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CF], check=True)
    subprocess.run(['chmod', '+x', CF], check=True)

# 只清理同 token 的旧连接，保留 quick tunnel 做回退
subprocess.run("ps -eo pid,args | grep '[c]loudflared tunnel.*run --token' | awk '{print $1}' | xargs -r kill", shell=True)
time.sleep(1)
LOG = '/content/cloudflared_named.log'
log = open(LOG, 'w')
p = subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--loglevel', 'info', 'run', '--token', TOKEN],
                     stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('Named Tunnel PID:', p.pid, '| 等待云端下发 ingress ...')
time.sleep(12)

API = 'https://qwythos.gmhqyj.ccwu.cc'
COMFY = 'https://comfy.gmhqyj.ccwu.cc'
for label, url in [('8010 /health', API + '/health'), ('8010 /v1/models', API + '/v1/models'),
                   ('8188 /system_stats', COMFY + '/system_stats'), ('8188 Web UI', COMFY + '/')]:
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                body = r.read()
                print('%-20s HTTP %s | %d bytes' % (label, r.status, len(body)))
            break
        except Exception as e:
            if attempt == 2:
                print('%-20s 失败: %r（DNS 未生效就等 30 秒重跑本 Cell）' % (label, e))
            else:
                time.sleep(8)
print('固定域名:', API, '|', COMFY)


In [ ]:
# 可选：给 8188 单独开一条 quick tunnel（本机打不开私有域名时用这个临时地址）
import subprocess, re, time, os, socket, urllib.request
PORT, CF, LOG = 8188, '/usr/local/bin/cloudflared', '/content/cloudflared_8188.log'

def alive(p):
    s = socket.socket(); s.settimeout(1)
    try:
        s.connect(('127.0.0.1', p)); return True
    except Exception:
        return False
    finally:
        s.close()

assert alive(PORT), '8188 未监听：先重跑「启动 ComfyUI」Cell'
subprocess.run("ps -eo pid,args | grep '[c]loudflared.*--url http://localhost:8188' | awk '{print $1}' | xargs -r kill", shell=True)
time.sleep(1)
log = open(LOG, 'w')
subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8188'],
                 stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

URL = None
for _ in range(40):
    time.sleep(2)
    m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open(LOG, errors='replace').read())
    if m:
        URL = m[0]; break
print('>>> ComfyUI 临时公网地址:', URL or '未取到，见 /content/cloudflared_8188.log')
if URL:
    for path in ['/', '/system_stats']:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(URL + path, timeout=45) as r:
                    print('  %-14s HTTP %s | %d bytes' % (path, r.status, len(r.read())))
                break
            except Exception as e:
                if attempt == 2:
                    print('  %-14s 失败: %r（新域名 DNS 未生效，等 30-60 秒重跑）' % (path, e))
                else:
                    time.sleep(10)
    print('Web UI:', URL)
    print('原生 API: %s/prompt  /history  /view  /object_info  wss://.../ws' % URL)


# 用你自己的 OpenAI 客户端调用

```python
from openai import OpenAI
client = OpenAI(base_url='https://qwythos.gmhqyj.ccwu.cc/v1', api_key='not-needed')
resp = client.images.generate(
    model='qwen-image-2.1-turbo-Q4_K_M.gguf',
    prompt='春日樱花下的江南小镇，水墨风格，8K',
    width=1024, height=1024
)
img_b64 = resp.data[0].b64_json
```

### FAQ
- **跑不起来？** 确认 Colab 运行时选了 **T4 GPU**；本地 CPU 推理 DiT 7B 太慢，不建议。
- **显存不足？** 把 `qwen3vl_8b_int8_convrot.safetensors` 换更小的量化，或把 `width/height` 降到 768。
- **下载慢？** Colab 机器在境外，pip 与模型都走官方源（`pypi.org` / `huggingface.co`）；
  国内镜像常未同步 2026 新包（如 `comfy-kitchen`），**不要**改用清华源或 hf-mirror。
- **ComfyUI 报 `cannot import name 'ck'`？** 缺 `comfy-kitchen==0.2.41`，用官方 PyPI 重装 `requirements.txt`。
- **模型文件 0 字节？** HF 仓库里 `text_encoders/` 和 `vae/` 是子目录，下载 URL 必须带上，否则 404。
- **公网域名 502？** 隧道活着但 8010 没监听。按顺序重跑「写入 qwen_api.py → 启动代理」两个 Cell；
  用 `curl http://127.0.0.1:8010/health` 确认后端，再刷新公网域名。隧道进程保留则域名不变。
- **不想要 FastAPI？** 把隧道 Cell 的 `TUNNEL_PORT` 改成 `8188`，直接用 ComfyUI 原生 `/prompt`、`/history`、`/view`。
- **固定私有域名：** Cloudflare Named Tunnel 上已配两条 Public Hostname：
  `qwythos.gmhqyj.ccwu.cc` → `localhost:8010`（OpenAI 兼容接口），
  `comfy.gmhqyj.ccwu.cc` → `localhost:8188`（ComfyUI Web UI / 原生 API）。
  两者都无鉴权，建议加 Cloudflare Access；若本地打不开先换网络/代理试，域名侧返回 200 即为链路问题。
- **接口风格：** 保留 OpenAI `images.generate` 兼容；`b64_json` 直接解码为图片。
- **测速指标：** 测试 Cell 打印单图端到端耗时（含 ComfyUI 排队与去噪），可用于对比不同量化版本。
- **新域名自检报 `gaierror(-2, Name or service not known)`？** quick tunnel 域名刚生成时 DNS 还没传播，
  不是服务挂了；等 30–60 秒重跑验证即可，可选单元格已内置 3 次重试。
- **本机打不开固定域名？** 部分网络会在 TLS 握手阶段阻断 `*.gmhqyj.ccwu.cc`，从 Colab / 其它外网出口
  却能正常 200；用可选单元格给出的 `trycloudflare.com` 临时域名，或换网络 / 调整代理规则。
